# Notebook 15 — Diagnostics round 2: is validation too easy? how much does the representation lose? what errors remain?
Solar Filament Segmentation Challenge 2026 — Kaggle edition. No training.

An independent reviewer questioned three of our conclusions. This notebook tests them directly:

**A. Is our validation split "too easy" compared with the test set?**
Dates were assigned to validation at random, so validation images may have training images from adjacent days, whereas the test images might be farther from training.
We compare, for validation and for test images, the distance in days to the nearest *training* image (plus station and year mix). (CPU only.)

**B. How much do we lose by training on the UNION of annotators' polygons and recovering instances with connected components?**
"Oracle" test: feed the exact ground-truth union (a perfect semantic prediction) through connected components, and through our real pipeline
(1024 px -> upsample -> threshold -> closing -> connected components -> minimum size), and score against each annotator record with the official rules.
This is a diagnostic reference for the *representation*, not a ceiling for any model. (CPU only.)

**C. What errors does our best train-only model (B04, last epoch) still make?**
Error anatomy under the official protocol (fragments, merges, boundary mismatches, misses, false positives that touch no annotation), plus pooled PQ by station and by year block.
(GPU, about 10 minutes.)

**Setup:** Add Input -> the competition AND the finished **B04** output (folder `B04_resnet50_s0`). GPU on. Internet not needed. Save & Run All (~30 min). Send me the **NOTEBOOK 15 SUMMARY** block.

In [4]:
# ============================================================
# CONFIGURATION
# ============================================================
KAGGLE_DATA_CANDIDATES = [
    "/kaggle/input/competitions/filament-segmentation-2026",
    "/kaggle/input/filament-segmentation-2026",
]

ENCODER = "resnet34"                    # default for the shared model cell (members choose their own)
TRAIN_SEED_OFFSET = 0                   # 0, 1, 2 ... (split is NOT affected)
assert ENCODER in ("resnet34", "resnet50", "resnet101")
EXPERIMENT = "D02_diagnostics"
RUN_MODE = "FULL"                       # names kept for the shared helper cells
DEBUG_MODE = False
OUTPUT_DIR = "/kaggle/working"
SEED = 42
RESUME = True

CFG = {
    "experiment_id": EXPERIMENT,
    "train_resolution": 1024,
    "batch_size": 4,                    # if the preflight reports CUDA out-of-memory, set 2 here and re-run
    "epochs": 50,
    "lr": 1e-3,                         # decoder learning rate
    "lr_encoder": 1e-4,                 # pretrained encoder learning rate
    "weight_decay": 1e-5,
    "model": f"{ENCODER}-UNet(ImageNet)",
    "encoder": ENCODER,
    "train_seed_offset": TRAIN_SEED_OFFSET,
    "loss": "BCE+Dice",
    "prob_threshold": 0.5,
    "min_instance_px": 0,
    "debug_n_train": 24,
    "debug_n_val": 8,
    "use_amp": True,
    "num_workers": None,
    "pq_eval_every": 10,
    "snapshot_every": 10,
    "ckpt_every": 2,
    "fsync_checkpoints": False,
    "multi_record": True,               # random annotator record per training sample (B02a)
    "cosine_lr": True,                  # cosine decay to 1% of the base learning rates
    "eta_min": 1e-5,                    # decoder end LR (encoder end LR scales proportionally)
    "bn_recal_batches": 60,
    "recipe": {"tta": True, "threshold": 0.5, "min_px": 400, "closing_r": 10},
}
print(f"EXPERIMENT={EXPERIMENT} | pretrained {ENCODER} encoder | seed offset {TRAIN_SEED_OFFSET} | multi_record={CFG['multi_record']} | cosine_lr={CFG['cosine_lr']} | epochs={CFG['epochs']}")

EXPERIMENT=D02_diagnostics | pretrained resnet34 encoder | seed offset 0 | multi_record=True | cosine_lr=True | epochs=50


In [5]:
import copy
import os, sys, json, random, re, warnings, math, shutil, time, hashlib, platform, gc, datetime
from pathlib import Path
from collections import defaultdict, Counter

try:
    import pycocotools
except ImportError:
    !pip -q install pycocotools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from pycocotools import mask as mask_utils

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---- GPU is REQUIRED. No silent CPU fallback. ----
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. This notebook will NOT fall back to CPU.\n"
        "Fix: in the Kaggle right-hand panel open Session options -> Accelerator -> choose GPU T4 x2 or P100, "
        "then run again.")
DEVICE = torch.device("cuda:0")
N_GPUS_VISIBLE = torch.cuda.device_count()
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | device {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)} | visible GPUs: {N_GPUS_VISIBLE} "
      f"| this notebook uses ONLY cuda:0 (no DataParallel)")

if CFG["num_workers"] is None:
    CFG["num_workers"] = min(4, os.cpu_count() or 2)

# ---- output location (Kaggle working dir) ----
OUT_ROOT = Path(OUTPUT_DIR); OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUT_ROOT / EXPERIMENT
OUT.mkdir(parents=True, exist_ok=True)
IN_KAGGLE = Path("/kaggle/input").exists()
print("Running on Kaggle:", IN_KAGGLE, "| Outputs:", OUT)
print("NOTE: /kaggle/working is kept only when you Save Version (Save & Run All); lost when the session ends otherwise.")

# ---- bring a previous version's checkpoints into /kaggle/working (resume across sessions) ----
RESUME_FILES = ["latest_checkpoint.pth", "latest_checkpoint_prev.pth", "baseline_best.pth",
                "baseline_best_meta.json", "training_history.csv", "split_metadata.csv"]

def import_previous_output():
    if (OUT / "latest_checkpoint.pth").exists() or not IN_KAGGLE:
        return
    hits = [p for p in Path("/kaggle/input").rglob("latest_checkpoint*.pth") if p.parent.name == OUT.name]
    if not hits:
        return
    src = max(hits, key=lambda p: p.stat().st_mtime).parent
    print(f">>> Found a previous run's checkpoints in {src} — copying them into {OUT} for resume")
    for f in RESUME_FILES + [p.name for p in src.glob("checkpoint_epoch_*.pth")]:
        if (src / f).exists() and not (OUT / f).exists():
            shutil.copyfile(src / f, OUT / f)

import_previous_output()
print("DataLoader workers:", CFG["num_workers"])

Python 3.13.16 | torch 2.11.0+cu130 | device cuda:0
GPU: Tesla T4 | visible GPUs: 2 | this notebook uses ONLY cuda:0 (no DataParallel)
Running on Kaggle: True | Outputs: /kaggle/working/D02_diagnostics
NOTE: /kaggle/working is kept only when you Save Version (Save & Run All); lost when the session ends otherwise.
DataLoader workers: 4


In [6]:
# ============================================================
# DATA DISCOVERY on Kaggle (/kaggle/input) — same discovery logic as Notebook 03
# ============================================================

def acquire_data_root():
    for c in KAGGLE_DATA_CANDIDATES:
        if Path(c).exists():
            return Path(c)
    base = Path("/kaggle/input")
    if base.exists() and any(base.rglob("train_images")):
        print("WARNING: standard path not found; using /kaggle/input and auto-discovery.")
        return base
    raise FileNotFoundError(
        "Competition data not found under /kaggle/input.\n"
        "Fix: right-hand panel -> Add Input -> search 'filament-segmentation-2026' -> Add.")

DATA_ROOT = acquire_data_root()
print("DATA_ROOT:", DATA_ROOT)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def discover_dataset(root):
    """Find train/test image dirs and the COCO annotation JSON. Fail loudly."""
    img_dirs = []
    for d in sorted({p.parent for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS}):
        n = sum(1 for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
        img_dirs.append((d, n))
        print(f"  image dir: {d.relative_to(root)}  ({n} images)")
    train_cands = [d for d, _ in img_dirs
                   if "train" in str(d).lower() and "test" not in str(d).lower()]
    test_cands = [d for d, _ in img_dirs
                  if "test" in str(d).lower() and "train" not in str(d).lower()]
    if len(train_cands) != 1 or len(test_cands) != 1:
        raise RuntimeError(
            f"Ambiguous dataset layout. train candidates={train_cands}, "
            f"test candidates={test_cands}. Inspect the listing above.")
    coco = None
    for p in root.rglob("*.json"):
        try:
            with open(p) as f:
                d = json.load(f)
            if isinstance(d, dict) and "images" in d and "annotations" in d:
                coco = p; break
        except Exception:
            pass
    if coco is None:
        raise RuntimeError(f"No COCO-style annotation JSON found under {root}")
    return train_cands[0], test_cands[0], coco

TRAIN_DIR, TEST_DIR, ANN_PATH = discover_dataset(DATA_ROOT)

train_paths = sorted(p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
test_paths  = sorted(p for p in TEST_DIR.iterdir()  if p.suffix.lower() in IMG_EXTS)
name_to_path = {p.name: p for p in train_paths}

print(f"\ntrain images: {len(train_paths)} | test images: {len(test_paths)}")
print(f"annotations: {ANN_PATH.name}")
assert len(train_paths) > 0 and len(test_paths) > 0

DATA_ROOT: /kaggle/input/competitions/filament-segmentation-2026
  image dir: MAGFiLO_1.0_Kaggle_2026/test/test_images  (180 images)
  image dir: MAGFiLO_1.0_Kaggle_2026/train/train_images  (707 images)

train images: 707 | test images: 180
annotations: MAGFiLO_1.0_Annotations_kaggle2026_train.json


In [7]:
# ============================================================
# ANNOTATIONS + BASELINE GT REPRESENTATION  (unchanged from Notebook 03)
# Baseline target = FIRST annotation record per file (sorted by id).
# ============================================================

def load_annotations(path):
    with open(path) as f:
        coco = json.load(f)
    images_info = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_image[a["image_id"]].append(a)
    return coco, images_info, anns_by_image

COCO, images_info, anns_by_image = load_annotations(ANN_PATH)
print(f"JSON records: {len(images_info)} | annotations: {len(COCO['annotations'])}")

file_to_records = defaultdict(list)
for img_id, info in images_info.items():
    fname = Path(info["file_name"]).name
    if fname in name_to_path:
        file_to_records[fname].append(img_id)

missing = len(images_info) - sum(len(v) for v in file_to_records.values())
if missing:
    raise RuntimeError(f"{missing} JSON records reference files not on disk")

file_to_primary = {f: sorted(ids)[0] for f, ids in file_to_records.items()}
primary_ids = sorted(file_to_primary.values())
print(f"physical files: {len(file_to_primary)} | primary records: {len(primary_ids)}")

def ann_to_mask(ann, h, w):
    m = np.zeros((h, w), np.uint8)
    for poly in ann["segmentation"]:
        pts = np.asarray(poly, np.float64).reshape(-1, 2)
        cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
    return m.astype(bool)

def build_gt_masks(img_id):
    """Full-resolution instance masks for one record."""
    info = images_info[img_id]
    masks = [ann_to_mask(a, info["height"], info["width"])
             for a in anns_by_image[img_id]]
    masks = [m for m in masks if m.any()]
    if not masks:
        raise ValueError(f"Record {img_id} has no non-empty GT masks")
    return masks

def load_image(path):
    img = np.array(Image.open(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    return img.astype(np.float32)

JSON records: 1154 | annotations: 8199
physical files: 707 | primary records: 707


In [8]:
# ============================================================
# OFFICIAL EVALUATOR REPRODUCTION
# Many-to-many matching: EVERY (pred, gt) pair with IoU > 0.5 (STRICT >)
# is a TP and contributes its IoU to the numerator.
# FP = pred with zero qualifying matches; FN = gt with zero.
# PQ = sum(TP IoUs) / (TP + 0.5 FP + 0.5 FN)
# Diagnostics (one-to-many / many-to-one) use overlap > 0.
#
# Only change vs Notebook 03: the pairwise IoU/Dice loop skips pairs whose
# bounding boxes do not intersect and only intersects the bbox overlap region.
# Results are mathematically identical (verified against the original
# implementation in the equivalence test below); it is just much faster.
# ============================================================

def _bbox(m):
    rows = np.flatnonzero(m.any(axis=1)); cols = np.flatnonzero(m.any(axis=0))
    if len(rows) == 0:
        return (0, 0, 0, 0)
    return (rows[0], rows[-1] + 1, cols[0], cols[-1] + 1)

def iou_and_dice_matrix(pred_masks, gt_masks):
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    pb = [_bbox(p) for p in pred_masks]; gb = [_bbox(g) for g in gt_masks]
    pa = [int(p.sum()) for p in pred_masks]; ga = [int(g.sum()) for g in gt_masks]
    for i, p in enumerate(pred_masks):
        py0, py1, px0, px1 = pb[i]
        for j, g in enumerate(gt_masks):
            gy0, gy1, gx0, gx1 = gb[j]
            y0, y1 = max(py0, gy0), min(py1, gy1)
            x0, x1 = max(px0, gx0), min(px1, gx1)
            if y0 >= y1 or x0 >= x1:
                continue
            inter = int(np.logical_and(p[y0:y1, x0:x1], g[y0:y1, x0:x1]).sum())
            if inter == 0:
                continue
            iou[i, j] = inter / (pa[i] + ga[j] - inter)
            dce[i, j] = 2 * inter / (pa[i] + ga[j])
    return iou, dce

def compute_official_pq(gt_masks, pred_masks):
    if len(gt_masks) == 0 and len(pred_masks) == 0:
        return {"PQ": 1.0, "TP": 0, "FP": 0, "FN": 0,
                "tp_ious": [], "tp_dices": [],
                "one_to_many": 0, "many_to_one": 0}
    iou, dce = iou_and_dice_matrix(pred_masks, gt_masks)
    hit = iou > 0.5                                   # STRICT >
    tp_ious = iou[hit].tolist()
    tp_dices = dce[hit].tolist()
    tp = len(tp_ious)
    fp = int((hit.sum(axis=1) == 0).sum()) if len(pred_masks) else 0
    fn = int((hit.sum(axis=0) == 0).sum()) if len(gt_masks) else 0
    denom = tp + 0.5 * fp + 0.5 * fn
    overlap = iou > 0
    one_to_many = int((overlap.sum(axis=0) > 1).sum())   # GT hit by multiple preds
    many_to_one = int((overlap.sum(axis=1) > 1).sum())   # pred hitting multiple GTs
    return {"PQ": (sum(tp_ious) / denom) if denom > 0 else 0.0,
            "TP": tp, "FP": fp, "FN": fn,
            "tp_ious": tp_ious, "tp_dices": tp_dices,
            "one_to_many": one_to_many, "many_to_one": many_to_one}

# ---------------- MANDATORY UNIT TESTS (same 7 as Notebook 03) ----------------
def _box(y0, y1, x0, x1, hw=64):
    m = np.zeros((hw, hw), bool); m[y0:y1, x0:x1] = True; return m

def run_metric_tests():
    results = []
    g = _box(10, 20, 10, 30)                       # 10x20 = 200 px

    r = compute_official_pq([g], [g.copy()])
    results.append(("T1 perfect -> PQ=1", abs(r["PQ"] - 1.0) < 1e-9))

    half = _box(10, 20, 10, 20)                    # IoU = 0.5 exactly
    r = compute_official_pq([g], [half])
    results.append(("T2 IoU==0.5 not TP", r["TP"] == 0 and r["FP"] == 1 and r["FN"] == 1))

    most = _box(10, 20, 10, 24)                    # IoU = 0.7
    r = compute_official_pq([g], [most])
    results.append(("T3 IoU>0.5 is TP", r["TP"] == 1 and r["FP"] == 0 and r["FN"] == 0))

    p1 = _box(10, 20, 10, 27); p2 = _box(10, 20, 12, 30)
    r = compute_official_pq([g], [p1, p2])
    results.append(("T4 1 GT + 2 preds -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    g1 = _box(10, 20, 10, 30); g2 = _box(10, 20, 8, 32); pred = _box(10, 20, 9, 31)
    r = compute_official_pq([g1, g2], [pred])
    results.append(("T5 2 GT + 1 pred -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    r = compute_official_pq([g], [])
    results.append(("T6 no preds -> FN=1, PQ=0", r["FN"] == 1 and r["PQ"] == 0.0))

    r = compute_official_pq([], [g])
    results.append(("T7 no GT -> FP=1, PQ=0", r["FP"] == 1 and r["PQ"] == 0.0))

    all_ok = True
    for name, ok in results:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
        all_ok &= ok
    if not all_ok:
        raise AssertionError("Metric unit tests FAILED — do not proceed")
    print("All 7 metric tests passed.")

run_metric_tests()

# ---------------- EQUIVALENCE TEST: fast matrix == original Notebook-03 matrix ----------------
def _iou_and_dice_matrix_reference(pred_masks, gt_masks):
    """Verbatim original implementation from Notebook 03 (slow, full-frame)."""
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    for i, p in enumerate(pred_masks):
        ps = p.sum()
        for j, g in enumerate(gt_masks):
            inter = np.logical_and(p, g).sum()
            if inter == 0:
                continue
            gs = g.sum()
            iou[i, j] = inter / (ps + gs - inter)
            dce[i, j] = 2 * inter / (ps + gs)
    return iou, dce

def run_equivalence_test(n_trials=40, hw=96):
    rng = np.random.RandomState(0)
    def rand_mask():
        m = np.zeros((hw, hw), np.uint8)
        for _ in range(rng.randint(1, 4)):
            y, x = rng.randint(0, hw - 8, 2); h, w = rng.randint(2, 30, 2)
            m[y:y + h, x:x + w] = 1
        return m.astype(bool)
    for _ in range(n_trials):
        P = [rand_mask() for _ in range(rng.randint(0, 6))]
        G = [rand_mask() for _ in range(rng.randint(0, 6))]
        a = iou_and_dice_matrix(P, G); b = _iou_and_dice_matrix_reference(P, G)
        assert np.allclose(a[0], b[0]) and np.allclose(a[1], b[1]), "fast IoU/Dice differs from original"
    print(f"Equivalence test passed: fast evaluator == original on {n_trials} random cases.")

run_equivalence_test()

  PASS  T1 perfect -> PQ=1
  PASS  T2 IoU==0.5 not TP
  PASS  T3 IoU>0.5 is TP
  PASS  T4 1 GT + 2 preds -> TP=2 FP=0 FN=0
  PASS  T5 2 GT + 1 pred -> TP=2 FP=0 FN=0
  PASS  T6 no preds -> FN=1, PQ=0
  PASS  T7 no GT -> FP=1, PQ=0
All 7 metric tests passed.
Equivalence test passed: fast evaluator == original on 40 random cases.


In [9]:
# ============================================================
# SPLIT — date-grouped 80/20 (Notebook 02 A3). Same function, same seed as Notebook 03.
# ============================================================

FN_RE = re.compile(r"^(\d{4})(\d{2})(\d{2})\d{6}([A-Za-z])h\.", re.I)

def create_split(primary_ids, val_frac=0.2, seed=SEED):
    rows = []
    for img_id in primary_ids:
        fname = Path(images_info[img_id]["file_name"]).name
        m = FN_RE.match(fname)
        if not m:
            raise ValueError(f"Unparseable filename: {fname}")
        rows.append({"img_id": img_id, "file": fname,
                     "date": f"{m[1]}-{m[2]}-{m[3]}", "station": m[4].upper()})
    df = pd.DataFrame(rows)
    dates = sorted(df["date"].unique())
    rng = random.Random(seed); rng.shuffle(dates)
    n_val_dates = int(round(len(dates) * val_frac))
    val_dates = set(dates[:n_val_dates])
    df["split"] = df["date"].map(lambda d: "val" if d in val_dates else "train")
    return df

split_df = create_split(primary_ids)
train_ids = split_df.loc[split_df.split == "train", "img_id"].tolist()
val_ids   = split_df.loc[split_df.split == "val", "img_id"].tolist()

if DEBUG_MODE:                       # smoke-test shortcut — must be OFF for Baseline-01
    train_ids = train_ids[:CFG["debug_n_train"]]
    val_ids   = val_ids[:CFG["debug_n_val"]]

SPLIT_HASH = hashlib.sha1(",".join(sorted(
    split_df.loc[split_df.split == "val", "file"])).encode()).hexdigest()[:12]

split_df.to_csv(OUT / "split_metadata.csv", index=False)
print(f"train: {len(train_ids)} | val: {len(val_ids)} (date-grouped) | split hash: {SPLIT_HASH}")
assert not set(split_df[split_df.split == "train"].date) & \
           set(split_df[split_df.split == "val"].date)

train: 568 | val: 139 (date-grouped) | split hash: 0cc2d729a5b1


In [10]:
import torchvision
import torch.nn.functional as F

# ============================================================
# MODEL — plain U-Net (base 32), UNCHANGED.  Loss — BCE + soft Dice, UNCHANGED
# (logits are cast to float32 inside the loss so AMP cannot affect the loss math).
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=1, base=32):
        super().__init__()
        self.e1, self.e2 = ConvBlock(cin, base), ConvBlock(base, base*2)
        self.e3, self.e4 = ConvBlock(base*2, base*4), ConvBlock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*8, base*16)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2); self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2);  self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2);  self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2);    self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        m = self.mid(self.pool(e4))
        d = self.d4(torch.cat([self.u4(m), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))
        return self.out(d)

ENCODER_SPECS = {
    "resnet34":  {"ctor": torchvision.models.resnet34,  "weights": torchvision.models.ResNet34_Weights.IMAGENET1K_V1,  "ch": [64, 64, 128, 256, 512]},
    "resnet50":  {"ctor": torchvision.models.resnet50,  "weights": torchvision.models.ResNet50_Weights.IMAGENET1K_V1,  "ch": [64, 256, 512, 1024, 2048]},
    "resnet101": {"ctor": torchvision.models.resnet101, "weights": torchvision.models.ResNet101_Weights.IMAGENET1K_V1, "ch": [64, 256, 512, 1024, 2048]},
}

class ResNetUNet(nn.Module):
    """U-Net with a ResNet-34 encoder (torchvision). Input: 1-channel image in [0,1]; the model repeats it to 3 channels and
    applies ImageNet mean/std normalisation internally, so the data pipeline stays unchanged."""
    def __init__(self, pretrained=False, encoder=None):
        super().__init__()
        spec = ENCODER_SPECS[encoder or CFG["encoder"]]; c = spec["ch"]
        weights = spec["weights"] if pretrained else None
        try:
            r = spec["ctor"](weights=weights)
        except Exception as e:
            raise RuntimeError("Could not download the ImageNet encoder weights. Turn Internet ON "
                               "(Session options -> Internet) and run again. Original error: " + repr(e))
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)                 # stride 2,  64 ch
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4   # /4:64  /8:128  /16:256  /32:512
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.d4 = ConvBlock(c[4] + c[3], 256)
        self.d3 = ConvBlock(256 + c[2], 128)
        self.d2 = ConvBlock(128 + c[1], 64)
        self.d1 = ConvBlock(64 + c[0], 64)
        self.d0 = ConvBlock(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def encoder_parameters(self):
        for mod in (self.stem, self.l1, self.l2, self.l3, self.l4):
            yield from mod.parameters()

    def decoder_parameters(self):
        for mod in (self.d4, self.d3, self.d2, self.d1, self.d0, self.out):
            yield from mod.parameters()

    def forward(self, x):
        x3 = (x.repeat(1, 3, 1, 1) - self.mean) / self.std
        s0 = self.stem(x3)                      # /2
        s1 = self.l1(self.pool(s0))             # /4
        s2 = self.l2(s1)                        # /8
        s3 = self.l3(s2)                        # /16
        s4 = self.l4(s3)                        # /32
        up = lambda t, ref: F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)
        d = self.d4(torch.cat([up(s4, s3), s3], 1))
        d = self.d3(torch.cat([up(d, s2), s2], 1))
        d = self.d2(torch.cat([up(d, s1), s1], 1))
        d = self.d1(torch.cat([up(d, s0), s0], 1))
        d = self.d0(F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False))
        return self.out(d)

def build_model(pretrained=False, encoder=None):
    return ResNetUNet(pretrained=pretrained, encoder=encoder).to(DEVICE)

def build_optimizer(model):
    groups = [{"params": list(model.encoder_parameters()), "lr": CFG["lr_encoder"], "base_lr": CFG["lr_encoder"]},
              {"params": list(model.decoder_parameters()), "lr": CFG["lr"], "base_lr": CFG["lr"]}]
    return torch.optim.Adam(groups, weight_decay=CFG["weight_decay"])

def bce_dice_loss(logits, target, eps=1.0):
    logits = logits.float()
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(2, 3))
    dice = (2 * inter + eps) / (prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps)
    return bce + (1 - dice).mean()

USE_AMP = bool(CFG["use_amp"]) and DEVICE.type == "cuda"

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

def build_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP)

_tmp_model = build_model()
N_PARAMS = sum(p.numel() for p in _tmp_model.parameters())
del _tmp_model
print(f"parameters: {N_PARAMS:,} | AMP: {USE_AMP}")

parameters: 24,522,785 | AMP: True


In [11]:
# ============================================================
# HELPERS: pooled-PQ summary, BN re-estimation, flip-TTA prediction, post-processing
# ============================================================
def summarize(items):
    tp = sum(r["TP"] for r in items); fp = sum(r["FP"] for r in items); fn = sum(r["FN"] for r in items)
    ious = [x for r in items for x in r["tp_ious"]]
    denom = tp + 0.5 * fp + 0.5 * fn
    return {"PQ_pooled": (sum(ious) / denom) if denom else 0.0,
            "PQ_mean_per_image": float(np.mean([r["PQ"] for r in items])) if items else 0.0,
            "TP": tp, "FP": fp, "FN": fn, "mean_tp_iou": float(np.mean(ious)) if ious else 0.0}


def load_model(path):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    m = build_model(); m.load_state_dict(sd); m.eval()
    return m

def recalibrate_bn(model, n_batches=60):
    """Re-estimate BatchNorm running statistics on ~240 training images (weights untouched)."""
    m = copy.deepcopy(model)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.reset_running_stats(); mod.momentum = None
    g = torch.Generator(); g.manual_seed(SEED)
    loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
                        drop_last=True, generator=g, worker_init_fn=_worker_init)
    it = iter(loader); m.train()
    with torch.no_grad():
        for _ in range(n_batches):
            b = next(it)
            with autocast_ctx():
                m(b["image"].to(DEVICE))
    del it; m.eval()
    return m

def postprocess(prob, thr, min_px, close_r):
    binary = (prob > thr).astype(np.uint8)
    if close_r:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_r + 1, 2 * close_r + 1)))
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    areas = np.bincount(labels.ravel(), minlength=n)
    return [labels == k for k in range(1, n) if areas[k] > min_px]

@torch.no_grad()
def predict_views(model, img_path, resolution, tta=True):
    """Returns (plain prob, flip-TTA-averaged prob) at model resolution + original image shape."""
    img = load_image(img_path)
    small = cv2.resize(img, (resolution, resolution), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    views = [None, [3], [2], [2, 3]] if tta else [None]
    outs = []
    for dims in views:
        xi = x if dims is None else torch.flip(x, dims)
        p = torch.sigmoid(model(xi))
        if dims is not None:
            p = torch.flip(p, dims)
        outs.append(p[0, 0].cpu().numpy())
    return outs[0], np.mean(outs, axis=0), img.shape

def upsample(p, shape):
    h, w = shape
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR)

# not used in this notebook (the weights are already BN-fixed); defined only so the shared helper above has no dangling names
train_ds = None
def _worker_init(_):
    cv2.setNumThreads(0)

In [12]:
# ============================================================
# OFFICIAL-PROTOCOL SCORING HELPERS (from Notebook 07): every annotator record of a file is scored separately and pooled
# ============================================================
vsplit = split_df[split_df.img_id.isin(val_ids)]
date_half = {d: "AB"[i % 2] for i, d in enumerate(sorted(vsplit.date.unique()))}
img_half = {r.img_id: date_half[r.date] for r in vsplit.itertuples()}

def records_of(img_id):
    fname = Path(images_info[img_id]["file_name"]).name
    return sorted(file_to_records[fname])

def entry_results(img_id, preds):
    """One result per annotation record of this file (records with no filaments do not exist in the official GT)."""
    out = []
    for rec in records_of(img_id):
        try:
            gt = build_gt_masks(rec)
        except ValueError:
            continue
        out.append(compute_official_pq(gt, preds))
    return out


def eval_config(prob_dict, thr, min_px, close_r, desc="eval"):
    prim, offi, halves = [], [], {"A": [], "B": []}
    for img_id in tqdm(val_ids, desc=desc, leave=False):
        p, shp = prob_dict[img_id]
        preds = postprocess(upsample(p, shp), thr, min_px, close_r)
        prim.append(compute_official_pq(build_gt_masks(img_id), preds))
        er = entry_results(img_id, preds)
        offi += er; halves[img_half[img_id]] += er
    return {"primary": summarize(prim), "official": summarize(offi),
            "official_A": summarize(halves["A"]), "official_B": summarize(halves["B"])}

In [13]:
# ============================================================
# ANALYSIS HELPERS (numpy only)
# ============================================================
AREA_BINS = [0, 50, 200, 1000, 5000, 10**12]
BIN_LABELS = ["<50 px", "50-199", "200-999", "1000-4999", ">=5000"]

def area_bin(a):
    for i in range(len(AREA_BINS) - 1):
        if AREA_BINS[i] <= a < AREA_BINS[i + 1]:
            return i

def analyze_image(gt, preds):
    """Anatomy of the official-metric result for one image (areas in 2048-px space)."""
    iou, _ = iou_and_dice_matrix(preds, gt)
    hit = iou > 0.5
    ov = iou > 0
    pa = [int(p.sum()) for p in preds]; ga = [int(g.sum()) for g in gt]
    pred_rows, gt_rows = [], []
    for i in range(len(preds)):
        is_fp = not hit[i].any()
        pred_rows.append({"area": pa[i], "bin": area_bin(pa[i]), "FP": is_fp,
                          "overlaps_gt": bool(ov[i].any()), "n_gt_overlapped": int(ov[i].sum())})
    for j in range(len(gt)):
        n_ov = int(ov[:, j].sum()) if len(preds) else 0
        is_fn = not (hit[:, j].any() if len(preds) else False)
        cat = "matched"
        union_iou = np.nan
        if is_fn:
            if n_ov == 0:
                cat = "missed: no overlapping prediction"
            elif n_ov == 1:
                cat = "missed: 1 overlapping prediction (IoU<=0.5)"
            else:
                u = np.zeros_like(gt[j])
                for i in np.flatnonzero(ov[:, j]):
                    u |= preds[i]
                union_iou = float((u & gt[j]).sum() / max((u | gt[j]).sum(), 1))
                cat = ("missed: fragmented, merge WOULD match (union IoU>0.5)" if union_iou > 0.5
                       else "missed: fragmented, merge would NOT match")
        gt_rows.append({"area": ga[j], "bin": area_bin(ga[j]), "FN": is_fn, "n_pred_overlapping": n_ov,
                        "category": cat, "union_iou": union_iou})
    return pred_rows, gt_rows

def summarize(items):
    """items: list of compute_official_pq results -> pooled PQ etc."""
    tp = sum(r["TP"] for r in items); fp = sum(r["FP"] for r in items); fn = sum(r["FN"] for r in items)
    ious = [x for r in items for x in r["tp_ious"]]
    denom = tp + 0.5 * fp + 0.5 * fn
    return {"PQ_pooled": (sum(ious) / denom) if denom else 0.0,
            "PQ_mean_per_image": float(np.mean([r["PQ"] for r in items])) if items else 0.0,
            "TP": tp, "FP": fp, "FN": fn,
            "mean_tp_iou": float(np.mean(ious)) if ious else 0.0}

In [14]:
# ============================================================
# A. IS VALIDATION "TOO EASY"?  distance (days) to the nearest TRAINING image, for validation vs test images; station and year mix
# ============================================================
import re, datetime
STEM_RE = re.compile(r"^(\d{4})(\d{2})(\d{2})\d{6}([A-Za-z])")
def parse_stem(stem):
    m = STEM_RE.match(stem)
    return np.datetime64(f"{m[1]}-{m[2]}-{m[3]}"), m[4].upper()

train_rows = split_df[split_df.split == "train"]
val_rows   = split_df[split_df.split == "val"]
train_d = np.array(sorted({parse_stem(Path(f).stem)[0] for f in train_rows.file}), dtype="datetime64[D]")
val_d   = [parse_stem(Path(f).stem)[0] for f in val_rows.file]
test_stems = [p.stem for p in test_paths]
test_d  = [parse_stem(s)[0] for s in test_stems]

def nearest_gap_days(dates):
    out = []
    for d in dates:
        i = np.searchsorted(train_d, d)
        c = []
        if i < len(train_d): c.append(abs((train_d[i] - d).astype(int)))
        if i > 0: c.append(abs((d - train_d[i - 1]).astype(int)))
        out.append(min(c))
    return np.array(out)

gv, gt_ = nearest_gap_days(val_d), nearest_gap_days(test_d)
def summ(g):
    return {"n": len(g), "median_days": float(np.median(g)), "mean_days": round(float(g.mean()), 1), "p90_days": float(np.percentile(g, 90)),
            "share_same_day": round(float((g == 0).mean()), 3), "share_<=1d": round(float((g <= 1).mean()), 3),
            "share_<=3d": round(float((g <= 3).mean()), 3), "share_<=7d": round(float((g <= 7).mean()), 3)}
gap_df = pd.DataFrame({"validation": summ(gv), "test": summ(gt_)})
print("Distance in days to the nearest TRAINING image (training = the 568 training-split files):"); print(gap_df.to_string())
print(f"\ntrain date range: {train_d.min()} .. {train_d.max()} | validation: {min(val_d)} .. {max(val_d)} | test: {min(test_d)} .. {max(test_d)}")

def mix(stems, name):
    s = pd.Series([parse_stem(x)[1] for x in stems]).value_counts(normalize=True).round(3)
    y = pd.Series([int(str(parse_stem(x)[0])[:4]) for x in stems]).value_counts(normalize=True).sort_index().round(3)
    return s.rename(name), y.rename(name)
tr_st, tr_y = mix([Path(f).stem for f in train_rows.file], "train"); va_st, va_y = mix([Path(f).stem for f in val_rows.file], "val"); te_st, te_y = mix(test_stems, "test")
print("\nstation share:"); print(pd.concat([tr_st, va_st, te_st], axis=1).fillna(0).to_string())
print("\nyear share:"); print(pd.concat([tr_y, va_y, te_y], axis=1).fillna(0).to_string())
gap_df.to_csv(OUT / "date_gap_audit.csv")

Distance in days to the nearest TRAINING image (training = the 568 training-split files):
                validation     test
n                  139.000  180.000
median_days          2.000    2.000
mean_days            4.100    3.000
p90_days             8.000    6.000
share_same_day       0.000    0.172
share_<=1d           0.317    0.450
share_<=3d           0.640    0.733
share_<=7d           0.871    0.922

train date range: 2011-01-09 .. 2022-08-03 | validation: 2011-01-19 .. 2021-10-16 | test: 2011-01-20 .. 2022-07-28

station share:
   train    val   test
C  0.178  0.173  0.183
M  0.178  0.173  0.144
L  0.176  0.194  0.189
B  0.169  0.158  0.183
T  0.162  0.129  0.150
U  0.137  0.173  0.150

year share:
      train    val   test
2011  0.118  0.137  0.133
2012  0.123  0.158  0.128
2013  0.148  0.122  0.128
2014  0.136  0.180  0.122
2015  0.125  0.108  0.094
2016  0.114  0.108  0.122
2017  0.107  0.072  0.106
2018  0.035  0.043  0.022
2019  0.018  0.029  0.011
2020  0.012  0.014  

In [15]:
# ============================================================
# B. REPRESENTATION ORACLE: perfect SEMANTIC foreground (the union of an annotator record's polygons) -> instances -> official PQ vs that record
#   O1  exact 2048-px union -> connected components (no closing, no size filter)
#   O2  exact union -> closing r=10 -> components -> drop <=400 px
#   O3  union through OUR pipeline (1024 nearest down-sampling -> bilinear up-sampling -> >0.5 -> components), no closing/filter
#   O4  union through OUR full recipe (O3 + closing r=10 + drop <=400 px)
# ============================================================
R = CFG["recipe"]
def cc_instances(binary, close_r=0, min_px=0):
    b = binary.astype(np.uint8)
    if close_r:
        b = cv2.morphologyEx(b, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_r + 1, 2 * close_r + 1)))
    n, labels = cv2.connectedComponents(b, connectivity=8)
    areas = np.bincount(labels.ravel(), minlength=n)
    return [labels == k for k in range(1, n) if areas[k] > min_px]

def through_our_pipeline(union):
    small = cv2.resize(union.astype(np.uint8), (CFG["train_resolution"],) * 2, interpolation=cv2.INTER_NEAREST).astype(np.float32)
    up = cv2.resize(small, (union.shape[1], union.shape[0]), interpolation=cv2.INTER_LINEAR)
    return up > 0.5

ORC = {"O1 exact union -> components": [], "O2 + closing + min size": [], "O3 our resolution path": [], "O4 our full recipe": []}
n_entries = 0
for img_id in tqdm(val_ids, desc="oracle"):
    for rec in records_of(img_id):
        try:
            gt = build_gt_masks(rec)
        except ValueError:
            continue
        U = np.zeros_like(gt[0])
        for g in gt: U |= g
        n_entries += 1
        ORC["O1 exact union -> components"].append(compute_official_pq(gt, cc_instances(U)))
        ORC["O2 + closing + min size"].append(compute_official_pq(gt, cc_instances(U, R["closing_r"], R["min_px"])))
        U1 = through_our_pipeline(U)
        ORC["O3 our resolution path"].append(compute_official_pq(gt, cc_instances(U1)))
        ORC["O4 our full recipe"].append(compute_official_pq(gt, cc_instances(U1, R["closing_r"], R["min_px"])))

rows = []
for name, items in ORC.items():
    s = summarize(items); rows.append({"oracle": name, "pooled_PQ": s["PQ_pooled"], "TP": s["TP"], "FP": s["FP"], "FN": s["FN"], "mean_tp_iou": s["mean_tp_iou"]})
oracle_df = pd.DataFrame(rows); oracle_df.to_csv(OUT / "oracle.csv", index=False)
print(f"annotator-record entries scored: {n_entries}")
print(oracle_df.round(4).to_string(index=False))
print("\nReading: a perfect SEMANTIC prediction passed through each pipeline. If O4 is far above our real PQ (0.44), instance extraction is not the bottleneck;"
      " if O1/O4 are low, merging/fragmenting caused by the union + connected-components representation limits any semantic model.")

oracle:   0%|          | 0/139 [00:00<?, ?it/s]

annotator-record entries scored: 225
                      oracle  pooled_PQ   TP  FP  FN  mean_tp_iou
O1 exact union -> components     0.9995 1606   0   1       0.9998
     O2 + closing + min size     0.8981 1510   7  97       0.9290
      O3 our resolution path     0.8881 1601   5   6       0.8911
          O4 our full recipe     0.8299 1517  13  90       0.8581

Reading: a perfect SEMANTIC prediction passed through each pipeline. If O4 is far above our real PQ (0.44), instance extraction is not the bottleneck; if O1/O4 are low, merging/fragmenting caused by the union + connected-components representation limits any semantic model.


In [16]:
# ============================================================
# C. ERROR ANATOMY OF THE BEST TRAIN-ONLY MODEL (B04 last epoch, recipe) under the official protocol, per annotator record
# ============================================================
def find_file(folder, fname):
    hits = [p for p in Path("/kaggle/input").rglob(fname) if p.parent.name == folder]
    if not hits:
        raise FileNotFoundError(f"{folder}/{fname} not found under /kaggle/input. Add the finished B04 output as an Input.")
    return max(hits, key=lambda p: p.stat().st_mtime)

wpath = find_file("B04_resnet50_s0", "last_epoch_bn_fixed.pth")
mdl = build_model(pretrained=False, encoder="resnet50")
mdl.load_state_dict(torch.load(wpath, map_location=DEVICE, weights_only=True)); mdl.eval()
print("loaded", wpath)

items, pred_rows, gt_rows, ent_rows = [], [], [], []
for img_id in tqdm(val_ids, desc="B04 last: predictions + anatomy"):
    fn = Path(images_info[img_id]["file_name"]).name
    _, tta_p, shp = predict_views(mdl, name_to_path[fn], CFG["train_resolution"], tta=True)
    preds = postprocess(upsample(tta_p, shp), R["threshold"], R["min_px"], R["closing_r"])
    stem = Path(fn).stem; station = stem[14].upper(); year = int(stem[:4])
    for rec in records_of(img_id):
        try:
            gt = build_gt_masks(rec)
        except ValueError:
            continue
        r = compute_official_pq(gt, preds); items.append(r)
        pr, gr = analyze_image(gt, preds)
        for x in pr: x.update(rec=rec, station=station, year=year)
        for x in gr: x.update(rec=rec, station=station, year=year)
        pred_rows += pr; gt_rows += gr
        ent_rows.append({"rec": rec, "station": station, "year": year, "n_gt": len(gt), "n_pred": len(preds), **{k: r[k] for k in ["TP", "FP", "FN"]},
                         "iou_sum": float(sum(r["tp_ious"]))})
    del preds
P, G, E = pd.DataFrame(pred_rows), pd.DataFrame(gt_rows), pd.DataFrame(ent_rows)
tot = summarize(items)
print(f"\nB04 last, recipe, official protocol: pooled PQ {tot['PQ_pooled']:.4f} | TP {tot['TP']} FP {tot['FP']} FN {tot['FN']} | mean TP IoU {tot['mean_tp_iou']:.3f}")

print("\nPREDICTION anatomy (all predicted instances x annotator records):")
fp = P[P.FP]
print(f"  total {len(P)} | false positives {len(fp)} ({len(fp)/len(P):.1%}): "
      f"{int((~fp.overlaps_gt).sum())} touch NO annotation of that record, {int(fp.overlaps_gt.sum())} overlap an annotation but IoU<=0.5")
print(f"  predictions that overlap >=2 different GT instances (possible MERGES): {int((P.n_gt_overlapped >= 2).sum())}")
print("\nGROUND-TRUTH anatomy (all annotator-record instances):")
miss = G[G.FN]
print(f"  total {len(G)} | missed {len(miss)} ({len(miss)/len(G):.1%})")
cat = miss.category.value_counts().rename_axis("category").reset_index(name="n"); cat["share_of_misses"] = (cat.n / len(miss)).round(3)
print(cat.to_string(index=False))
print(f"  ground-truth instances overlapped by >=2 predictions (FRAGMENTATION), matched or not: {int((G.n_pred_overlapping >= 2).sum())}")

def pooled_by(df, col):
    out = []
    for k, g in E.groupby(col):
        tp, fp_, fn = int(g.TP.sum()), int(g.FP.sum()), int(g.FN.sum()); den = tp + 0.5 * fp_ + 0.5 * fn
        out.append({col: k, "records": len(g), "pooled_PQ": round(g.iou_sum.sum() / den, 4) if den else 0.0, "TP": tp, "FP": fp_, "FN": fn})
    return pd.DataFrame(out)
E["year_block"] = pd.cut(E.year, [2010, 2013, 2016, 2019, 2022], labels=["2011-13", "2014-16", "2017-19", "2020-22"])
st_df, yb_df = pooled_by(E, "station"), pooled_by(E, "year_block")
print("\nPooled PQ by station (summed statistics, not a mean of per-image PQ):"); print(st_df.to_string(index=False))
print("\nPooled PQ by year block:"); print(yb_df.to_string(index=False))
size_rows = []
for b, lab in enumerate(BIN_LABELS):
    pb = P[P.bin == b]; gb = G[G.bin == b]
    size_rows.append({"area_bin": lab, "n_pred": len(pb), "pred_FP_rate": round(pb.FP.mean(), 3) if len(pb) else np.nan,
                      "n_gt": len(gb), "gt_FN_rate": round(gb.FN.mean(), 3) if len(gb) else np.nan})
size_df = pd.DataFrame(size_rows); print("\nBy instance area (2048-px pixels):"); print(size_df.to_string(index=False))
cat.to_csv(OUT / "gt_miss_categories.csv", index=False); st_df.to_csv(OUT / "pq_by_station.csv", index=False); yb_df.to_csv(OUT / "pq_by_year_block.csv", index=False)

loaded /kaggle/input/notebooks/parsabehjattabrizi/notebook-10-b04/B04_resnet50_s0/last_epoch_bn_fixed.pth


B04 last: predictions + anatomy:   0%|          | 0/139 [00:00<?, ?it/s]


B04 last, recipe, official protocol: pooled PQ 0.4435 | TP 1063 FP 577 FN 544 | mean TP IoU 0.677

PREDICTION anatomy (all predicted instances x annotator records):
  total 1640 | false positives 577 (35.2%): 212 touch NO annotation of that record, 365 overlap an annotation but IoU<=0.5
  predictions that overlap >=2 different GT instances (possible MERGES): 35

GROUND-TRUTH anatomy (all annotator-record instances):
  total 1607 | missed 544 (33.9%)
                                             category   n  share_of_misses
                    missed: no overlapping prediction 243            0.447
          missed: 1 overlapping prediction (IoU<=0.5) 220            0.404
missed: fragmented, merge WOULD match (union IoU>0.5)  56            0.103
            missed: fragmented, merge would NOT match  25            0.046
  ground-truth instances overlapped by >=2 predictions (FRAGMENTATION), matched or not: 95

Pooled PQ by station (summed statistics, not a mean of per-image PQ):
station 

In [17]:
# ============================================================
# SUMMARY — copy this block back to Claude
# ============================================================
L = []; add = L.append
add("NOTEBOOK 15 SUMMARY — diagnostics round 2 (no training)")
add("-" * 70)
add("A. nearest-TRAINING-image distance in days (validation vs test):")
for k in ["median_days", "mean_days", "p90_days", "share_same_day", "share_<=1d", "share_<=3d", "share_<=7d"]:
    add(f"   {k:<16} val {gap_df.loc[k, 'validation']} | test {gap_df.loc[k, 'test']}")
add("B. oracle (perfect semantic prediction through each pipeline): " + "; ".join(f"{r.oracle} {r.pooled_PQ:.3f}" for r in oracle_df.itertuples()))
add(f"C. B04 last, recipe, official: PQ {tot['PQ_pooled']:.4f} (TP {tot['TP']} FP {tot['FP']} FN {tot['FN']})")
add(f"   FP: {len(fp)} total, {int((~fp.overlaps_gt).sum())} touch no annotation, {int(fp.overlaps_gt.sum())} overlap with IoU<=0.5; predictions overlapping >=2 GT: {int((P.n_gt_overlapped >= 2).sum())}")
add("   misses: " + "; ".join(f"{r.category}: {r.n} ({r.share_of_misses:.0%})" for r in cat.itertuples()))
add("   PQ by station: " + ", ".join(f"{r.station} {r.pooled_PQ:.3f}" for r in st_df.itertuples()))
add("   PQ by year block: " + ", ".join(f"{r.year_block} {r.pooled_PQ:.3f}" for r in yb_df.itertuples()))
rep = "\n".join(L); print(rep); (OUT / "notebook15_report.txt").write_text(rep)
print("\nSaved to", OUT, ":", sorted(p.name for p in OUT.iterdir()))

NOTEBOOK 15 SUMMARY — diagnostics round 2 (no training)
----------------------------------------------------------------------
A. nearest-TRAINING-image distance in days (validation vs test):
   median_days      val 2.0 | test 2.0
   mean_days        val 4.1 | test 3.0
   p90_days         val 8.0 | test 6.0
   share_same_day   val 0.0 | test 0.172
   share_<=1d       val 0.317 | test 0.45
   share_<=3d       val 0.64 | test 0.733
   share_<=7d       val 0.871 | test 0.922
B. oracle (perfect semantic prediction through each pipeline): O1 exact union -> components 0.999; O2 + closing + min size 0.898; O3 our resolution path 0.888; O4 our full recipe 0.830
C. B04 last, recipe, official: PQ 0.4435 (TP 1063 FP 577 FN 544)
   FP: 577 total, 212 touch no annotation, 365 overlap with IoU<=0.5; predictions overlapping >=2 GT: 35
   misses: missed: no overlapping prediction: 243 (45%); missed: 1 overlapping prediction (IoU<=0.5): 220 (40%); missed: fragmented, merge WOULD match (union IoU>0.5): 